# Prepare VAD CSV for emotion2vec Wagner-compatible notebook

`vad_wagner_emotion2vec_experiment.ipynb` が読むCSV形式へ、手元のアノテーションCSVを変換します。出力形式は `file_path,valence,arousal,dominance,split,session` です。

## Target schema

必須列は `file_path` と、`arousal` / `dominance` / `valence` のうち少なくとも1列です。VAD値は Wagner 互換の `0..1` にそろえます。`dominance` がないデータセットでは空欄のままで問題ありません。

In [ ]:
# このセルでは、入力CSV・音声ルート・出力先などの変換設定を定義する。

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'vad_downstream':
    ROOT = ROOT.parent

# ---- edit these values for your source CSV ----
RAW_CSV = ROOT / 'data' / 'raw_vad_labels.csv'
OUTPUT_CSV = ROOT / 'data' / 'vad_labels_prepared.csv'

# Set this when file paths in RAW_CSV are relative to a specific audio directory.
# Leave as None if the source file column already contains paths usable from project root.
AUDIO_ROOT = None  # example: ROOT / 'audio'

# Map target column names to source column names. Use None when a label is unavailable.
COLUMN_MAP = {
    'file_path': 'file_path',
    'arousal': 'arousal',
    'dominance': 'dominance',
    'valence': 'valence',
    'split': 'split',
    'session': 'session',
}

# Supported: 'auto', 'zero_one', 'one_five', 'one_nine', 'minus_one_one'.
LABEL_SCALE = 'auto'

# Used only when the source CSV does not have a split column.
CREATE_SPLIT = True
SEED = 42
VAL_RATIO = 0.1
TEST_RATIO = 0.1

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
print(f'ROOT      : {ROOT}')
print(f'RAW_CSV   : {RAW_CSV}')
print(f'OUTPUT_CSV: {OUTPUT_CSV}')


In [ ]:
# このセルでは、VAD列の正規化やsplit/session付与に使う補助関数を定義する。

TARGET_COLUMNS = ['file_path', 'valence', 'arousal', 'dominance', 'split', 'session']
VAD_COLUMNS = ['valence', 'arousal', 'dominance']


def _get_source_column(df, target_name):
    source_name = COLUMN_MAP.get(target_name)
    if source_name is None:
        return None
    if source_name not in df.columns:
        if target_name == 'file_path':
            raise ValueError(f'{target_name!r} is mapped to missing source column {source_name!r}. Available columns: {list(df.columns)}')
        print(f'warning: {target_name!r} mapped to missing source column {source_name!r}; leaving it empty or auto-generated.')
        return None
    return source_name


def normalize_vad(values, scale='auto'):
    series = pd.to_numeric(values, errors='coerce')
    finite = series[np.isfinite(series)]
    if finite.empty:
        return series

    selected = scale
    if scale == 'auto':
        low = float(finite.min())
        high = float(finite.max())
        if 0.0 <= low and high <= 1.0:
            selected = 'zero_one'
        elif 1.0 <= low and high <= 5.0:
            selected = 'one_five'
        elif 1.0 <= low and high <= 9.0:
            selected = 'one_nine'
        elif -1.0 <= low and high <= 1.0:
            selected = 'minus_one_one'
        else:
            raise ValueError(f'Cannot infer VAD scale from range [{low}, {high}]. Set LABEL_SCALE explicitly.')

    if selected == 'zero_one':
        normalized = series
    elif selected == 'one_five':
        normalized = (series - 1.0) / 4.0
    elif selected == 'one_nine':
        normalized = (series - 1.0) / 8.0
    elif selected == 'minus_one_one':
        normalized = (series + 1.0) / 2.0
    else:
        raise ValueError("LABEL_SCALE must be one of: auto, zero_one, one_five, one_nine, minus_one_one")

    return normalized.clip(0.0, 1.0)


def make_splits(n_rows, seed=42, val_ratio=0.1, test_ratio=0.1):
    rng = np.random.default_rng(seed)
    indices = np.arange(n_rows)
    rng.shuffle(indices)

    split = np.full(n_rows, 'train', dtype=object)
    if n_rows < 3:
        return split

    n_test = max(1, int(round(n_rows * test_ratio))) if test_ratio > 0 else 0
    n_val = max(1, int(round(n_rows * val_ratio))) if val_ratio > 0 and n_rows >= 4 else 0
    if n_test + n_val >= n_rows:
        n_test = 1
        n_val = 0

    split[indices[:n_test]] = 'test'
    split[indices[n_test:n_test + n_val]] = 'val'
    return split


def prepare_vad_csv(raw_df):
    out = pd.DataFrame()

    file_col = _get_source_column(raw_df, 'file_path')
    if file_col is None:
        raise ValueError('COLUMN_MAP["file_path"] is required.')
    file_paths = raw_df[file_col].astype(str).str.strip()
    if AUDIO_ROOT is not None:
        file_paths = file_paths.apply(lambda value: str(Path(AUDIO_ROOT) / value) if value and not Path(value).is_absolute() else value)
    out['file_path'] = file_paths

    for label_name in VAD_COLUMNS:
        source_col = _get_source_column(raw_df, label_name)
        out[label_name] = '' if source_col is None else normalize_vad(raw_df[source_col], LABEL_SCALE)

    if out[VAD_COLUMNS].replace('', np.nan).isna().all(axis=1).any():
        bad_rows = out.index[out[VAD_COLUMNS].replace('', np.nan).isna().all(axis=1)].tolist()[:10]
        raise ValueError(f'Rows without any VAD label found: {bad_rows}')

    split_col = _get_source_column(raw_df, 'split')
    if split_col is not None:
        out['split'] = raw_df[split_col].astype(str).str.strip().str.lower()
    elif CREATE_SPLIT:
        out['split'] = make_splits(len(out), seed=SEED, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO)
    else:
        out['split'] = ''

    session_col = _get_source_column(raw_df, 'session')
    out['session'] = '' if session_col is None else raw_df[session_col].astype(str).str.strip()

    return out[TARGET_COLUMNS]


In [ ]:
# このセルでは、元CSVを読み込み、Wagner互換のVAD学習CSV形式へ変換する。

raw_df = pd.read_csv(RAW_CSV, encoding='utf-8-sig')
print(f'raw shape: {raw_df.shape}')
display(raw_df.head())

prepared_df = prepare_vad_csv(raw_df)
print(f'prepared shape: {prepared_df.shape}')
display(prepared_df.head())


In [ ]:
# このセルでは、変換後CSVの欠損値・値域・split分布を確認する。

print('missing values:')
display(prepared_df.replace('', np.nan).isna().sum())

for label_name in VAD_COLUMNS:
    values = pd.to_numeric(prepared_df[label_name], errors='coerce')
    finite = values[np.isfinite(values)]
    if not finite.empty:
        print(f'{label_name:9s}: n={len(finite):5d}, min={finite.min():.4f}, max={finite.max():.4f}, mean={finite.mean():.4f}')

if 'split' in prepared_df.columns:
    print('split counts:')
    display(prepared_df['split'].value_counts(dropna=False))

missing_audio = []
for path in prepared_df['file_path'].head(20):
    audio_path = Path(path)
    if not audio_path.is_absolute():
        audio_path = ROOT / audio_path
    if not audio_path.exists():
        missing_audio.append(str(audio_path))
if missing_audio:
    print('First rows include paths that do not exist from this notebook environment:')
    for path in missing_audio[:10]:
        print('  ', path)
else:
    print('First 20 audio paths exist.')


In [ ]:
# このセルでは、検証済みの変換結果をCSVとして保存する。

prepared_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'saved: {OUTPUT_CSV}')


In [ ]:
# このセルでは、保存したCSVを実際のDataLoaderで読み込み、学習コードから使えるか確認する。

import sys

VAD_DIR = ROOT / 'vad_downstream'
if str(VAD_DIR) not in sys.path:
    sys.path.insert(0, str(VAD_DIR))

from data import WAGNER_VAD_COLUMNS, load_vad_csv

records = load_vad_csv(str(OUTPUT_CSV), audio_dir=str(ROOT))
label_counts = {name: int(np.isfinite([record[name] for record in records]).sum()) for name in WAGNER_VAD_COLUMNS}
print(f'loaded records: {len(records)}')
print(f'label counts  : {label_counts}')
print(f'first record   : {records[0]}')
